In [1]:
from pathlib import Path
import polars as pl
import yaml

RUN = Path("/s/project/ml4rg_students/2026/project15/working/binding_bench_runs/supervised_small_cnn")
DS = RUN / "binding_bench/discrete/DNA_rossi_chipexo"

conf_path = DS / "benchmark_conf.yaml"
with open(conf_path) as f:
    conf = yaml.safe_load(f)

conf["status"], conf.get("message")

('SUCCESS', None)

In [2]:
raw = pl.read_parquet(DS / "full_discrete_metrics_ds_DNA_rossi_chipexo.parquet")

best_jaccard = pl.read_parquet(
    DS / "full_best_assnt/best_assnt_metrics_jaccard_ds_DNA_rossi_chipexo.parquet"
)
best_precision = pl.read_parquet(
    DS / "full_best_assnt/best_assnt_metrics_precision_lb_ds_DNA_rossi_chipexo.parquet"
)
best_recall = pl.read_parquet(
    DS / "full_best_assnt/best_assnt_metrics_recall_lb_ds_DNA_rossi_chipexo.parquet"
)

raw.shape, raw.columns

((12312, 19),
 ['feature_idx',
  'name',
  'TP_s',
  'TP_p',
  'n_overlap',
  'sites_at_k',
  'peaks_at_k',
  'sites_total',
  'peaks_total',
  'precision',
  'recall',
  'jaccard',
  'f1',
  'precision_lb',
  'precision_ub',
  'recall_lb',
  'recall_ub',
  'sites_frac_bg',
  'prec_lb_fc'])

In [3]:
diagonal = raw.filter(pl.col("feature_idx") == pl.col("name"))

diagonal.select(
    pl.mean("jaccard").alias("mean_jaccard"),
    pl.mean("precision_lb").alias("mean_precision_lb"),
    pl.mean("recall_lb").alias("mean_recall_lb"),
    pl.median("jaccard").alias("median_jaccard"),
    pl.len().alias("n_tfs"),
)

mean_jaccard,mean_precision_lb,mean_recall_lb,median_jaccard,n_tfs
f64,f64,f64,f64,u32
0.022599,0.121307,0.017212,0.018182,154


In [4]:
diagonal.select(
    "name", "jaccard", "precision_lb", "recall_lb", "sites_at_k", "peaks_at_k"
).sort("jaccard", descending=True).head(20)

name,jaccard,precision_lb,recall_lb,sites_at_k,peaks_at_k
str,f64,f64,f64,i64,i64
"""mif2""",0.0703125,0.197821,0.079764,137,137
"""ssn8""",0.064516,0.363507,0.034033,33,33
"""brn1""",0.0625,0.033002,0.033002,34,34
"""spt8""",0.058824,0.231424,0.031116,36,36
"""spt20""",0.054945,0.257699,0.034698,48,48
…,…,…,…,…,…
"""hsf1""",0.042802,0.189159,0.041692,134,134
"""smc4""",0.042254,0.247542,0.017044,37,37
"""hap3""",0.042169,0.191285,0.044948,173,173


In [5]:
diagonal.select(
    "name", "jaccard", "precision_lb", "recall_lb", "sites_at_k", "peaks_at_k"
).sort("jaccard").head(20)

name,jaccard,precision_lb,recall_lb,sites_at_k,peaks_at_k
str,f64,f64,f64,i64,i64
"""cdc46""",0.002088,0.000105,0.000105,240,240
"""rna14""",0.003618,0.020091,0.002903,971,971
"""swd3""",0.005405,0.011842,0.000272,93,93
"""bdf1""",0.005858,0.042302,0.004695,601,601
"""yta7""",0.006225,0.037158,0.006193,889,889
…,…,…,…,…,…
"""aor1""",0.010526,0.000527,0.000527,48,48
"""sfh1""",0.010526,0.002533,0.002533,96,96
"""rad6""",0.010601,0.059909,0.004347,143,143
